# ML-03 — Frame Your Lane as an ML Task

## Context

For this FlyRank lane, I frame the problem around identifying content pages that are likely to decline so a content/SEO team can prioritize review and refresh work. The goal is the decision, not the model itself.


## 1. My lane as an ML task (type)

**Task type: Classification.**

The question is: **will this content page be observed as declining?** That is a binary outcome, so classification is the clearest framing. The output supports a practical decision: which pages should the content/SEO team review first.

A wrong call has an asymmetric operational cost: a missed declining page can delay a useful refresh, while reviewing a page that turns out not to be declining mainly costs analyst/editor time.


In [1]:
task_type = "classification"
print(f"Task type: {task_type.title()}")


Task type: Classification


## 2. Target or proxy

**Target: `is_declining_label`.**

The starter pipeline defines this observed label as 1 when `trend_direction == "down"`, and 0 otherwise. This is an observed outcome in the supplied slice rather than a label invented from the model's own prediction.

**Leakage warning:** `trend_direction` and `trend_pct` are label-source columns, so they must not be used as model features. The target is defined from them, which would otherwise let the model see the answer.


In [2]:
target = "is_declining_label"
print(f"Target column: {target}")
print("Leakage columns excluded from features: trend_direction, trend_pct")


Target column: is_declining_label
Leakage columns excluded from features: trend_direction, trend_pct


## 3. Success metric

**Primary success metric: Recall.**

The decision is to surface pages that need attention. Missing a genuinely declining page is more costly than sending some extra pages for review, so recall is a defensible primary metric for the first version. I would report precision alongside recall in later modeling because editor time is also a real cost, but recall is the main success criterion for this framing.

The metric is defined before training: a better model is one that catches more of the observed declining pages without relying on the leaked label-source columns.


In [3]:
primary_metric = "Recall"
print(f"Primary metric: {primary_metric}")
print("Reason: prioritize catching genuinely declining pages")


Primary metric: Recall
Reason: prioritize catching genuinely declining pages


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one row = one pseudonymized content item/page.**

The starter data contains 30,000 rows and 44 columns. The loading cell below works both from the repository structure and from Google Colab by locating the local file first and then falling back to the repository's raw GitHub copy. IDs are used to identify/group records, not as predictive features.


In [4]:
from pathlib import Path
import pandas as pd
import urllib.request

# Try common local locations first (repo execution, notebook execution, or Colab).
candidates = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/ml-internship-2026/data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)

# In Colab, use the public raw copy from this repository if the repo is not cloned.
if DATA_PATH is None:
    DATA_PATH = Path("/tmp/content_refresh_anonymized.csv")
    DATA_URL = (
        "https://raw.githubusercontent.com/engyusufayman06/"
        "ml-internship-2026/main/data/raw/content_refresh_anonymized.csv"
    )
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
print("Loaded from:", DATA_PATH)
print("Shape:", df.shape)
print("Unit of analysis: one row = one content item/page")
display(df.head(10))


Loaded from: ../../data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)
Unit of analysis: one row = one content item/page


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


In [5]:
assert df.shape == (30000, 44), f"Unexpected starter shape: {df.shape}"
assert df["content_id"].nunique() == len(df), "Expected one row per content item"
if target not in df.columns:
    df[target] = (df["trend_direction"] == "down").astype(int)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Unique content_id values: {df['content_id'].nunique():,}")
print("Target values:")
print(df[target].value_counts().sort_index())


Rows: 30,000
Columns: 45
Unique content_id values: 30,000
Target values:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64


## 5. Why ML beats a fixed rule

A single threshold rule such as `impressions_last_30d < impressions_prev_30d` is easy to write, but it throws away the wider context available for each page. Decline risk can depend on a combination of search demand, clicks, sessions, CTR, search position, content age, update recency, keyword context, and content type. These signals can interact, and their useful relationships do not have to share one hand-written threshold.

ML earns its place if it can learn a repeatable pattern across those observed signals and improve the decision of **which pages the content/SEO team should review first**. If a simple rule performs just as well, then the rule is preferable; ML is not automatically better. For this phase, the claim is therefore **decision-support**, not an assertion that a model will definitely outperform a rule before it is tested.


In [6]:
decision = "prioritize content pages for review/refresh"
actor = "content/SEO team"
claim_level = "decision-support"
print(f"Decision supported: {decision}")
print(f"Who acts: {actor}")
print(f"Claim level: {claim_level}")


Decision supported: prioritize content pages for review/refresh
Who acts: content/SEO team
Claim level: decision-support


## Self-check

- [x] Task type is named and justified: classification.
- [x] Target/proxy is named: `is_declining_label`.
- [x] Success metric is named before modeling: recall.
- [x] Unit of analysis is demonstrated with the real starter dataframe: one row per content item/page.
- [x] The output is tied to a real content action: prioritize review/refresh work.
- [x] The ML-vs-rule argument is conditional and honest: ML must earn its place by improving the decision.
- [x] Label-source leakage columns are explicitly excluded: `trend_direction`, `trend_pct`.
- [x] No client names, URLs, or private queries are included.
